In [125]:
"""
<RAG 챗봇 서비스 흐름>
  - 사용자로부터 질문을 요청 받아서(Request)
  - 외부 데이터베이스(벡터 DB)에서 질문과 유사한 정보를 검색(Retrieval)하고,
  - LLM 모델에 의해 검색된 데이터를 기반으로(Augmented) 답변을 생성(Generation)하여,
  - 사용자에게 보다 정확하고 신뢰성 있는 응답을 제공함(Response)


<RAG 챗봇 개발 순서>
 * RAG 챗봇은 [인덱싱 과정]과 [쿼리 과정]으로 구분하여 개발함
   
 * 인덱싱 과정(Index Process)
    1. 문서를 수집하기
       - WEB 데이터, PDF, CSV 등 데이터 수집
    2. 검색이 용이하도록 벡터 데이터베이스에 저장하기
       - Chroma DB, FAISS DB 등에 저장


    ** >>> 이미 이전 파일에서 모두 진행 했음
   
 * 쿼리 과정(Query Process, 실제 서비스 과정)
    1. 사용자의 질문을 입력으로 요청 받기 (Request)
    2. 요청 질문과 유사한 정보를 실시간으로 검색하여 (Retrieval-Augmented)
    3. 검색된 데이터를 기반으로 답변을 생성하여 (Generation)
    4. 생성된 답변을 -> 사용자에게 응답하기 (Response)
"""


'\n<RAG 챗봇 서비스 흐름>\n  - 사용자로부터 질문을 요청 받아서(Request)\n  - 외부 데이터베이스(벡터 DB)에서 질문과 유사한 정보를 검색(Retrieval)하고,\n  - LLM 모델에 의해 검색된 데이터를 기반으로(Augmented) 답변을 생성(Generation)하여,\n  - 사용자에게 보다 정확하고 신뢰성 있는 응답을 제공함(Response)\n\n\n<RAG 챗봇 개발 순서>\n * RAG 챗봇은 [인덱싱 과정]과 [쿼리 과정]으로 구분하여 개발함\n   \n * 인덱싱 과정(Index Process)\n    1. 문서를 수집하기\n       - WEB 데이터, PDF, CSV 등 데이터 수집\n    2. 검색이 용이하도록 벡터 데이터베이스에 저장하기\n       - Chroma DB, FAISS DB 등에 저장\n\n\n    ** >>> 이미 이전 파일에서 모두 진행 했음\n   \n * 쿼리 과정(Query Process, 실제 서비스 과정)\n    1. 사용자의 질문을 입력으로 요청 받기 (Request)\n    2. 요청 질문과 유사한 정보를 실시간으로 검색하여 (Retrieval-Augmented)\n    3. 검색된 데이터를 기반으로 답변을 생성하여 (Generation)\n    4. 생성된 답변을 -> 사용자에게 응답하기 (Response)\n'

In [126]:
"""
<인덱싱 과정>
 1. PDF 파일 로드
 2. 텍스트 분할
 3. 크로마(Chroma) or FAISS Vector DB에 저장시키기


<쿼리 과정>
 1. 프로프트 템플릿 정의
 2. LLM 생성형 모델 정의
 3. 크로마(Chroma) DB에서 검색을 위한 설정 : as_retriever() 함수 사용
 4. 문맥 구성 : DB에서 검색된 데이터를 줄바꿈 처리
 5. 체인 구성 : RAG 검색 | 프롬프트 템플릿 | LLM 모델 | 출력
 6. 대화 기록 메모리 설정
 7. 챗봇 서비스 실행(사용자와의 인터페이스)
"""

'\n<인덱싱 과정>\n 1. PDF 파일 로드\n 2. 텍스트 분할\n 3. 크로마(Chroma) or FAISS Vector DB에 저장시키기\n\n\n<쿼리 과정>\n 1. 프로프트 템플릿 정의\n 2. LLM 생성형 모델 정의\n 3. 크로마(Chroma) DB에서 검색을 위한 설정 : as_retriever() 함수 사용\n 4. 문맥 구성 : DB에서 검색된 데이터를 줄바꿈 처리\n 5. 체인 구성 : RAG 검색 | 프롬프트 템플릿 | LLM 모델 | 출력\n 6. 대화 기록 메모리 설정\n 7. 챗봇 서비스 실행(사용자와의 인터페이스)\n'

#### 사용할 라이브러리 정의

In [127]:
### [인덱싱 과정]에서 주로 사용되는 라이브러리
# - PDF 문서 로더 클래스(여러가지 있음)
from langchain_community.document_loaders import PyPDFLoader

# - 임베딩 클래스(문자를 숫자로 변환)
from langchain_ollama.embeddings import OllamaEmbeddings

# - 재귀적 텍스트 분할기 클래스(여러가지 있음)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# - 사용할 Vector DB 클래스
from langchain_chroma import Chroma

### [쿼리 과정]에서 주로 사용되는 라이브러리
# - 프롬프트 템플릿 클래스
from langchain_core.prompts import ChatPromptTemplate

# - Ollama 클래스 : 채팅 API
from langchain_ollama import ChatOllama

# - 출력 파서 클래스
from langchain_core.output_parsers import StrOutputParser

# - 체인(chain) 내에서 입력 값을 수정 없이 다음 단계(기능)로 그대로 전달하는 클래스
#   -> LLM의 출력을 정리하여 보기 좋게 일반 텍스트로 처리하는데 사용
from langchain_core.runnables import RunnablePassthrough

### 대화의 이력(메시지 기록)을 저장하고 관리하는 라이브러리
# - 사용자의 질문과 AI 응답을 순서대로 기록하여 대화의 맥락을 유지하기 위한 클래스
# (구버전 : from langchain.memory import ChatMessageHistory)
from langchain_community.chat_message_histories import ChatMessageHistory

# - 대화 이력 관리를 통합하는 클래스
#   -> LLM 체인에 대화 기록을 자동으로 삽입하고, 응답 후 기록을 업데이트하는 클래스
#   -> 상태를 저장하고 대화하는 기능을 구현
from langchain_core.runnables import RunnableWithMessageHistory


from langchain_community.vectorstores import FAISS

In [128]:
"""
<인덱싱 과정>
 1. PDF 파일 로드
 2. 텍스트 분할
 3. 크로마(Chroma) or FAISS Vector DB에 저장시키기
"""

'\n<인덱싱 과정>\n 1. PDF 파일 로드\n 2. 텍스트 분할\n 3. 크로마(Chroma) or FAISS Vector DB에 저장시키기\n'

In [129]:
# 1. PDF 파일 로드
# - pages 변수에 pdf 파일 읽어들여 주세요.
loder = PyPDFLoader("./Data/2024_KB_부동산_보고서_최종.pdf")
pages = loder.load()


In [130]:
# 2. 텍스트 분할
# - texts 변수에 읽어들인 데이터 분할하기(분할 사이즈 1000, 중복 200)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

texts = text_splitter.split_documents(pages)
print(f"{len(texts)}")

135


In [131]:
# 3. 크로마(Chroma) DB에 저장시키기

embedding_function = OllamaEmbeddings(model="nomic-embed-text")

save_path = "./Chroma_DB/"

vectordb = Chroma.from_documents(
    documents = texts,
    embedding = embedding_function,
    persist_directory = save_path
)
vectordb

In [132]:
### 최종 DB에 저장된 청크 데이터 갯수
vectordb._collection.count()

270

In [133]:
"""
<가정>
- Chroma DB를 불러들였다고 가정하고 진행..
- 실제 서비스시에는 불러들인 후 아래 진행..

<쿼리 과정>
 1. 프로프트 템플릿 정의
 2. LLM 생성형 모델 정의
 3. 크로마(Chroma) DB에서 검색을 위한 설정 : as_retriever() 함수 사용
 4. 문맥 구성 : DB에서 검색된 데이터를 줄바꿈 처리
 5. 체인 구성 : RAG 검색 | 프롬프트 템플릿 | LLM 모델 | 출력
 6. 대화 기록 메모리 설정
 7. 챗봇 서비스 실행(사용자와의 인터페이스)
"""

'\n<가정>\n- Chroma DB를 불러들였다고 가정하고 진행..\n- 실제 서비스시에는 불러들인 후 아래 진행..\n\n<쿼리 과정>\n 1. 프로프트 템플릿 정의\n 2. LLM 생성형 모델 정의\n 3. 크로마(Chroma) DB에서 검색을 위한 설정 : as_retriever() 함수 사용\n 4. 문맥 구성 : DB에서 검색된 데이터를 줄바꿈 처리\n 5. 체인 구성 : RAG 검색 | 프롬프트 템플릿 | LLM 모델 | 출력\n 6. 대화 기록 메모리 설정\n 7. 챗봇 서비스 실행(사용자와의 인터페이스)\n'

In [134]:
### 1. 프롬프트 템플릿 정의

#  - 챗봇(system) 역할 정의
#  - context : 검색(RAG) 결과를 체인을 통해 받을 때 내부적으로 사용되는 변수명
system_role = """
    너는 지금부터 DB 부동산 보고서 작성 전문가 입니다.
    다음 정보를 바탕으로, 사용자의 질문에 답변 합니다.
    컨텍스트 : {context}
"""

# - 프롬프트 템플릿 정의하기
prompt = ChatPromptTemplate.from_messages([
    # 시스템 역할 정의
    ("system", system_role),
    # 과거의 대화 이력이 삽입될 위치 지정(위치 지정자)
    #  - LangChain 내부적으로 메시지를 관리하는데만 사용됨
    #  - chat_history : history_messages_key에 있는 값
    ("placeholder", "{chat_history}"),
    # 사용자로부터 입력을 받은 질문이 포함되는 영역 정의
    #  - 사용자 역할은 user 또는 human 모두 사용가능
    ("human", "{question}")
])
prompt

ChatPromptTemplate(input_variables=['context', 'question'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='System

In [135]:
# 2. LLM 생성형 모델 정의
# - 창의성 제어 : 0.1로, 응답 문자 길이 : 100자리, 사용모델 : mistral
# - 변수명 : model
model = ChatOllama(
    model = "mistral",
    temperature = 0.1,
    max_token = 100
)
model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)

In [136]:
# 3. 크로마(Chroma) DB에서 검색을 위한 설정 : as_retriever() 함수 사용
# - LangChain에서 사용하는 검색함수는 : as_retriever(), 체인 연결이 가능한 함수
retriever = vectordb.as_retriever(
    ### 검색 방법 정의
    # - 문서 검색 시 "가장 비슷한 순서대로" 조회할 때 사용(가장 기본적으로 사용됨)
    # - 유사도 점수(score)의 기준선(threshold)을 통과한 문서만 가져오겠다는 의미
    search_type = "similarity_score_threshold",

    ### 검색 규칙 정의
    # - score_threshold : 사용자의 질문과 문서가 수학적으로 얼마나 비슷한지를 계산
    #                   : 코사인 유사도 점수가 0.5(50%) 이상인 문서만 검색하겠다는 의미
    # - k(kwargs) : 코사인 유사도 점수가 0.5(50%) 이상인 문서들 중에 상위 3개 문서(청크)만 조회
    search_kwargs = {"score_threshold" : 0.5, "k" : 3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7306c2453190>, search_type='similarity_score_threshold', search_kwargs={'score_threshold': 0.5, 'k': 3})

In [137]:
# 4. 문맥 구성 : DB에서 검색된 데이터를 줄바꿈 처리(리스트를 -> 문자열로 변환처리하는 함수)
#  - docs : 검색결과 3건의 리스트을 받는 매개변수
def format_docs(docs) :
    # join() : 리스트 내의 각 데이터를 문자열로 합칠 때 사용하는 함수
    # \n\n : 각 데이터를 문자열로 합칠 때 줄바꿈을 넣어서 합치겠다는 의미
    # 예시 결과값 : "1번청크문서\n\n2번청크문서\n\n3번청크문서"
    all_str = "\n\n".join(doc.page_content for doc in docs)
    return all_str

In [138]:
# 5. 체인 구성 : RAG 검색 | 프롬프트 템플릿 | LLM 모델 | 출력
# 출력 파서 정의
parser = StrOutputParser()
"""
<체인구성순서>
1. Vector DB 검색 기능  : 검색된 문서는 하나의 문자열로 처리해서 다음 기능으로 넘겨야 함.
2. 프롬프트 템플릿 기능
3. 생성형 모델
4. 출력 파서
"""

chain = (
    # 1. Vector DB 검색 기능
    #  - Vector DB에서 검색한 정보를 손실 없이 그대로 다음 프롬프트 템플릿으로 전달
    RunnablePassthrough.assign(
        # 검색 결과를 하나의 문자열로 처리해서 context 속성 변수에 넣어 주어야 함.
        #  - lambda : 함수 호출 방법 중 한 가지
        #  - x : 함수가 처리한 결과값(문자열)을 받아올 변수
        #  - format_docs() : 우리가 정의할 함수(리스트를 문자열로 변환하는 함수)
        #  - x["question"] : 사용자의 질문에 대한 검색 결과를 함수에 전달
        #                  : 최종 함수의 리턴 결과를 x변수가 자동으로 받아옴
        context = lambda x: format_docs(retriever.invoke(x["question"]))
    )

    # 2.프롬프트 템플릿 기능
    | prompt

    # 3. 생성형 모델
    | model

    # 4. 출력 파서
    | parser
)

In [139]:
# 6. 대화 기록 메모리 설정
# - 대화기록 유지를 위한 메모리 클래스 생성
chat_history = ChatMessageHistory()

# - 챗봇 대화 중에 기록을 유지하기 위한 설정
# - 세션 관리 클래스 설정
chain_with_memory = RunnableWithMessageHistory(
    # 체인(chain) 정의
    chain,

    # 세션 ID별로 대화기록 생성을 위한 세션ID Key 정의
    lambda session_id : chat_history,

    # 질문 Key 정의
    input_messages_key = "question",

    # 대화기록 key 정의
    history_messages_key = "chat_history"
)
chain_with_memory

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  chat_history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function <lambda> at 0x7307cb566c20>, input_messages_key='question', history_messages_key='chat_history', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [140]:
# 7. 챗봇 서비스 실행(사용자와의 인터페이스)

# - 세션 아이디 정의 : 사용자의 대화를 식별하는데 사용됨
session_id = "user_session"

# - 챗봇 서비스 최초 실행 시 안내 문구 출력
print("안녕하세요. 부동산 보고서 챗봇입니다. 질문해 주세요!! (종료는 'quit' 입력!)")

### 실시간 반복 수행
while True :
    # - 사용자 질문 입력처리
    user_input = input("[사용자] : ")
    print(f"[사용자] : {user_input}")

    # - 반복 종료 처리(강제 종료)
    if user_input.lower() in ["quit", "q"]:
        print("[챗봇] : 다음에 또 만나요~~")
        break

    elif user_input == "" :
        print("[질문] : 질문을 입력하셔야 합니다. (종료는 'quit' or 'q' 입력)")
        continue

    # - 사용자의 질문에 대한 답변 생성
    response = chain_with_memory.invoke(
        ### 질문 전달 정의
        {"question" : user_input},

        ### 사용자 history 관리를 위해 식별자(세션 ID) 전달
        # - configurable 이름과 session_id 키(key) dlfmadms
        #  -> LangChain의 메모리(대화기록) 관련 클래스에서
        #  -> 내부적으로 사용하는 정의된 공식적인 이름입니다.
        {"configurable" : {"session_id" : session_id}}
    )

    # - 생성된 답변 출력
    print(f"[챗봇] : {response}")

안녕하세요. 부동산 보고서 챗봇입니다. 질문해 주세요!! (종료는 'quit' 입력!)
[사용자] : 부동산시장
[챗봇] :  사용자의 질문에 답변합니다.

1. 이 보고서는 누구가 발행했나요?
    - 이 보고서는 KB금융지주 경영연구소에서 발행했습니다.

2. 이 보고서의 발행인은 누구입니까?
    - 이 보고서의 발행인은 양종희입니다.

3. 이 보고서의 편집인은 누구입니까?
    - 이 보고서의 편집인은 황원경입니다.

4. 이 보고서가 발행된 곳은 어디입니까?
    - 이 보고서는 KB금융지주 경영연구소(서울시 영등포구 국제금융로8길 26, 12층)에서 발행되었습니다.

5. 이 보고서에 연구자 팀장은 누구입니까?
    - 이 보고서의 연구자 팀장은 강민석입니다.

6. 이 보고서에 선임연구위원들은 누구입니까?
    - 이 보고서에 선임연구위원들은 손은경, 김태환, 김진성, 이강배입니다.

7. 이 보고서에 연구위원은 누구입니까?
    - 이 보고서에 연구위원은 정종훈입니다.

8. 이 보고서의 주제는 무엇입니까?
    - 이 보고서의 주제는 "2024 KB부동산 보고서: 상업용 부동산시장 진단과 전망"입니다.

9. 이 보고서에서 무점포소매업에 대해 어떤 내용이 있나요?
    - 이 보고서에서는 인터넷 쇼핑이 호조를 보이면서 성장을 주도하고 있다고 설명합니다. 홈쇼핑이나 방문 배달 및 소매점은 횡보세를 보이나 인터넷 쇼핑은 가파른 성장세가 멈추지 않고 있다고 합니다. 코로나팬데믹 이후 소비 패턴 변화로 온라인 소비가 주요 소비 트렌드로 자리 잡은 셈이며, 온라인 소비 문화가 정착되면서 판매 상품이 다양해지고 있으며, 스마트폰을 통한 결제 서비스의 편리함까지 더해지면서 시장 성장이 가속화되고 있는 것으로 판단된다고 합니다.

10. 이 보고서에서 상가 임대료에 대해 어떤 내용이 있나요?
    - 이 보고서에서는 상가 임대료는 코로나팬데믹이 끝난 이후에도 지속적으로 하락하고 있다고 설명합니다. 대부분의 상가 규모에서 임대료지수가 하락하

No relevant docs were retrieved using the relevance score threshold 0.5


[사용자] : 1


KeyboardInterrupt: 

In [ ]:
"""(실습)
* 선택 질문

- 첫번째 입력 받기
  "모델을 선택하여 주세요(번호 입력) :
    1번 mistral
    2번 qwen2.5:0.5b"

  

- 두번쨰 입력 받기
  "Vector DB를 선택하여 주세요(번호 입력) :
    1번 Chroma Vector DB
    2번 FAISS Vector DB

- 세번째 입력 받기 (계속 반복)
  "질문을 입력하여 주세요!(종료는 quit 또는 q 입력!)"
"""

In [148]:
### 1. 프롬프트 템플릿 정의

#  - 챗봇(system) 역할 정의
#  - context : 검색(RAG) 결과를 체인을 통해 받을 때 내부적으로 사용되는 변수명
system_role = """
    너는 지금부터 DB 부동산 보고서 작성 전문가 입니다.
    다음 정보를 바탕으로, 사용자의 질문에 답변 합니다.
    컨텍스트 : {context}
"""

# - 프롬프트 템플릿 정의하기
prompt = ChatPromptTemplate.from_messages([
    # 시스템 역할 정의
    ("system", system_role),
    # 과거의 대화 이력이 삽입될 위치 지정(위치 지정자)
    #  - LangChain 내부적으로 메시지를 관리하는데만 사용됨
    #  - chat_history : history_messages_key에 있는 값
    ("placeholder", "{chat_history}"),
    # 사용자로부터 입력을 받은 질문이 포함되는 영역 정의
    #  - 사용자 역할은 user 또는 human 모두 사용가능
    ("human", "{question}")
])
prompt

ChatPromptTemplate(input_variables=['context', 'question'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='System

In [149]:
#첫번째 입력 받기
#  "모델을 선택하여 주세요(번호 입력) :
#    1번 mistral
#    2번 qwen2.5:0.5b"

def choose_model():
    input_model = int(input("""모델을 선택하여 주세요(번호 입력) :
                            1번 mistral,
                            2번 qwen2.5:0.5b"""))
    if input_model == 1:
        input_model = "mistral"
    elif input_model == 2:
        input_model = "qwen2.5:0.5b"

    model = ChatOllama(
        model = input_model,
        temperature = 0.1,
        max_token = 100
    )
    return model

ch_m = choose_model()
ch_m

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='qwen2.5:0.5b', temperature=0.1)

In [150]:
# 두번째 입력 받기
#  "Vector DB를 선택하여 주세요(번호 입력) :
#    1번 Chroma Vector DB
#    2번 FAISS Vector DB

def choose_DB():
    input_DB = int(input("""Vector DB를 선택하여 주세요(번호 입력) :
                            1번 : Chroma Vector DB,
                            2번 : FAISS Vector DB"""))

    embedding_function = OllamaEmbeddings(model="nomic-embed-text")

    if input_DB == 1:
        vectordb = Chroma(
            embedding_function = embedding_function,
            persist_directory = "./Chroma_DB/"
        )
        return vectordb
    elif input_DB == 2:
        vectordb = FAISS.load_local(
            embeddings = embedding_function,
            folder_path = "./FAISS_DB/",
            ### 보안 경고 무시
            # - FAISS는 실행 시 보안 경고가 자주 출력됨
            allow_dangerous_deserialization = True
        )
        return vectordb

ch_db = choose_DB()
print(ch_db)


In [151]:
def search_set(p_vectordb):
    # 3. 크로마(Chroma) DB에서 검색을 위한 설정 : as_retriever() 함수 사용
    # - LangChain에서 사용하는 검색함수는 : as_retriever(), 체인 연결이 가능한 함수
    retriever = p_vectordb.as_retriever(
        ### 검색 방법 정의
        # - 문서 검색 시 "가장 비슷한 순서대로" 조회할 때 사용(가장 기본적으로 사용됨)
        # - 유사도 점수(score)의 기준선(threshold)을 통과한 문서만 가져오겠다는 의미
        search_type = "similarity_score_threshold",

        ### 검색 규칙 정의
        # - score_threshold : 사용자의 질문과 문서가 수학적으로 얼마나 비슷한지를 계산
        #                   : 코사인 유사도 점수가 0.5(50%) 이상인 문서만 검색하겠다는 의미
        # - k(kwargs) : 코사인 유사도 점수가 0.5(50%) 이상인 문서들 중에 상위 3개 문서(청크)만 조회
        search_kwargs = {"score_threshold" : 0.5, "k" : 3}
    )
    return retriever

search = search_set(ch_db)


In [152]:
# 5. 체인 구성 : RAG 검색 | 프롬프트 템플릿 | LLM 모델 | 출력

"""
<체인구성순서>
1. Vector DB 검색 기능  : 검색된 문서는 하나의 문자열로 처리해서 다음 기능으로 넘겨야 함.
2. 프롬프트 템플릿 기능
3. 생성형 모델
4. 출력 파서
"""
def format_docs(docs) :
    # join() : 리스트 내의 각 데이터를 문자열로 합칠 때 사용하는 함수
    # \n\n : 각 데이터를 문자열로 합칠 때 줄바꿈을 넣어서 합치겠다는 의미
    # 예시 결과값 : "1번청크문서\n\n2번청크문서\n\n3번청크문서"
    all_str = "\n\n".join(doc.page_content for doc in docs)
    print(f"\n[🔍 RAG 디버그: 검색된 관련 문서 {len(docs)}건을 바탕으로 답변합니다]")
    return all_str


def chain_set(p_prompt, p_model, p_retriever):
    chain = (
        # 1. Vector DB 검색 기능
        #  - Vector DB에서 검색한 정보를 손실 없이 그대로 다음 프롬프트 템플릿으로 전달
        RunnablePassthrough.assign(
            # 검색 결과를 하나의 문자열로 처리해서 context 속성 변수에 넣어 주어야 함.
            #  - lambda : 함수 호출 방법 중 한 가지
            #  - x : 함수가 처리한 결과값(문자열)을 받아올 변수
            #  - format_docs() : 우리가 정의할 함수(리스트를 문자열로 변환하는 함수)
            #  - x["question"] : 사용자의 질문에 대한 검색 결과를 함수에 전달
            #                  : 최종 함수의 리턴 결과를 x변수가 자동으로 받아옴
            context = lambda x: format_docs(p_retriever.invoke(x["question"]))
        )

        # 2.프롬프트 템플릿 기능
        | p_prompt

        # 3. 생성형 모델
        | p_model

        # 4. 출력 파서
        | StrOutputParser()
    )

    return chain

ch_set = chain_set(prompt, ch_m, search)


In [153]:
def memory_set(p_chain):
    # 6. 대화 기록 메모리 설정
    # - 대화기록 유지를 위한 메모리 클래스 생성
    chat_history = ChatMessageHistory()

    # - 챗봇 대화 중에 기록을 유지하기 위한 설정
    # - 세션 관리 클래스 설정
    chain_with_memory = RunnableWithMessageHistory(
        # 체인(chain) 정의
        p_chain,

        # 세션 ID별로 대화기록 생성을 위한 세션ID Key 정의
        lambda session_id : chat_history,

        # 질문 Key 정의
        input_messages_key = "question",

        # 대화기록 key 정의
        history_messages_key = "chat_history"
    )
    return chain_with_memory

mry_set = memory_set(ch_set)
print(mry_set)

bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  chat_history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]) kwargs={} config={} config_factories=[] get_session_history=<function memory_set.<locals>.<lambda> at 0x7307cbfb30a0> input_messages_key='question' history_messages_key='chat_history' history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)]


/tmp/ipykernel_65489/3111489996.py:23: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  mry_set = memory_set(ch_set)


In [154]:
def chat_service(p_chain_with_memory) :
    # 7. 챗봇 서비스 실행(사용자와의 인터페이스)

    # - 세션 아이디 정의 : 사용자의 대화를 식별하는데 사용됨
    session_id = "user_session"

    # - 챗봇 서비스 최초 실행 시 안내 문구 출력
    print("안녕하세요. 부동산 보고서 챗봇입니다. 질문해 주세요!! (종료는 'quit' 입력!)")

    ### 실시간 반복 수행
    while True :
        # - 사용자 질문 입력처리
        user_input = input("[사용자] : ")
        print(f"[사용자] : {user_input}")

        # - 반복 종료 처리(강제 종료)
        if user_input.lower() in ["quit", "q"]:
            print("[챗봇] : 다음에 또 만나요~~")
            break

        elif user_input == "" :
            print("[질문] : 질문을 입력하셔야 합니다. (종료는 'quit' or 'q' 입력)")
            continue

        # - 사용자의 질문에 대한 답변 생성
        response = p_chain_with_memory.invoke(
            ### 질문 전달 정의
            {"question" : user_input},

            ### 사용자 history 관리를 위해 식별자(세션 ID) 전달
            # - configurable 이름과 session_id 키(key) dlfmadms
            #  -> LangChain의 메모리(대화기록) 관련 클래스에서
            #  -> 내부적으로 사용하는 정의된 공식적인 이름입니다.
            {"configurable" : {"session_id" : session_id}}
        )

        # - 생성된 답변 출력
        print(f"[챗봇] : {response}")

chat_service(mry_set)


안녕하세요. 부동산 보고서 챗봇입니다. 질문해 주세요!! (종료는 'quit' 입력!)
[사용자] : 부동산

[🔍 RAG 디버그: 검색된 관련 문서 3건을 바탕으로 답변합니다]
[챗봇] : 부동산은 주택와 상업용 부동산 등 다양한 형태로 분류됩니다. 주택는 주택가, 상업용 부동산, 공원 등에 속하며, 상업용 부동산은 주택가와 상업용 공원에 속합니다. 

부동산 보고서는 주택 시장의 전망과 전반적인 상황을 파악하고, 주택 시장의 성장과 감소를 예측하는 데 도움이 될 수 있습니다. 

부동산 보고서는 주택 시장의 전반적인 상황을 파악하고, 주택 시장의 성장과 감소를 예측하는 데 도움이 될 수 있습니다. 

부동산 보고서는 주택 시장의 전반적인 상황을 파악하고, 주택 시장의 성장과 감소를 예측하는 데 도움이 될 수 있습니다. 

부동산 보고서는 주택 시장의 전반적인 상황을 파악하고, 주택 시장의 성장과 감소를 예측하는 데 도움이 될 수 있습니다. 

부동산 보고서는 주택 시장의 전반적인 상황을 파악하고, 주택 시장의 성장과 감소를 예측하는 데 도움이 될 수 있습니다. 

부동산 보고서는 주택 시장의 전반적인 상황을 파악하고, 주택 시장의 성장과 감소를 예측하는 데 도움이 될 수 있습니다. 

부동산 보고서는 주택 시장의 전반적인 상황을 파악하고, 주택 시장의 성장과 감소를 예측하는 데 도움이 될 수 있습니다. 

부동산 보고서는 주택 시장의 전반적인 상황을 파악하고, 주택 시장의 성장과 감소를 예측하는 데 도움이 될 수 있습니다. 

부동산 보고서는 주택 시장의 전반적인 상황을 파악하고, 주택 시장의 성장과 감소를 예측하는 데 도움이 될 수 있습니다. 

부동산 보고서는 주택 시장의 전반적인 상황을 파악하고, 주택 시장의 성장과 감소를 예측하는 데 도움이 될 수 있습니다. 

부동산 보고서는 주택 시장의 전반적인 상황을 파악하고, 주택 시장의 성장과 감소를 예측하는 데 도움이 될 수 있습니다. 

부동산 보고서는 주택 시장의 전반적인 상황을 파악하고, 주택 시장의

In [ ]:
ch_m
ch_db
search
ch_set
chat_service(mry_set)

안녕하세요. 부동산 보고서 챗봇입니다. 질문해 주세요!! (종료는 'quit' 입력!)


No relevant docs were retrieved using the relevance score threshold 0.5


[사용자] : ㅂ
[챗봇] : 죄송합니다, 저는 인공지능으로 인식된 자료를 제공하고 있습니다. 하지만, 현재의 부동산 시장의 연혁을 제공하는 것은 불가능합니다. 

다만, 현재의 부동산 시장의 연혁을 확인하고, 연혁별 부동산의 추이를 살펴보는 것은 가능합니다. 

1. **2020년 1월 1일부터 2021년 12월 31일까지**: 이 시기에는 부동산 시장은 급속한 성장세를 보였습니다. 
2. **2022년 1월 1일부터 2023년 12월 31일까지**: 이 시기에는 부동산 시장은 급속한 하락세를 보였습니다. 
3. **2024년 1월 1일부터 2025년 12월 31일까지**: 이 시기에는 부동산 시장은 급속한 상승세를 보였습니다.

이러한 연혁을 통해, 현재의 부동산 시장의 추이를 살펴보는 것은 가능합니다.
[사용자] : 부동산 가격
[챗봇] : 죄송합니다, 저는 인공지능으로 인식된 자료를 제공하고 있습니다. 하지만, 현재의 부동산 가격을 제공하는 것은 불가능합니다. 

다만, 현재의 부동산 가격을 확인하고, 가격의 변화를 살펴보는 것은 가능합니다. 

1. **2020년 1월 1일부터 2021년 12월 31일까지**: 이 시기에는 부동산 시장은 급속한 성장세를 보였습니다. 
2. **2022년 1월 1일부터 2023년 12월 31일까지**: 이 시기에는 부동산 시장은 급속한 하락세를 보였습니다. 
3. **2024년 1월 1일부터 2025년 12월 31일까지**: 이 시기에는 부동산 시장은 급속한 상승세를 보였습니다.

이러한 연혁을 통해, 현재의 부동산 시장의 추이를 살펴보는 것은 가능합니다.
[사용자] : 가격 추이
[챗봇] : 죄송합니다, 저는 인공지능으로 인식된 자료를 제공하고 있습니다. 하지만, 현재의 부동산 가격을 제공하는 것은 불가능합니다. 

다만, 현재의 부동산 가격을 확인하고, 가격의 변화를 살펴보는 것은 가능합니다. 

1. **2020년 1월 1일부터 2021년 12월 31일까지**: 이 시기에는 부동산 시장은 급속한 성장세를 보였습니다